# Objectives and workflow

This notebook evaluates a core-periphery network model on the public US air traffic dataset.

**Outputs**
- estimated parameters by period
- network metrics and diagnostic time series
- run artifacts stored under `results/` and `figures/`

**Workflow summary**
1. Load monthly snapshots and define period range
2. Select core size \(k\)
3. Estimate \(y\) and core node fields \(x_i\)
4. Run Monte Carlo simulations and diagnostics
5. Export tables and figures

**Model summary**
- undirected, unweighted graph with no self-loops
- \(p_{ij} = \sigma(y + \mathbb{1}_{i\in C}x_i + \mathbb{1}_{j\in C}x_j)\)
- independent edges conditional on \((y,\{x_i\})\)


## Data loading and period construction

Load preprocessed monthly snapshots from `data/processed/us_air/` and optionally aggregate to monthly, quarterly, or annual periods.
The tensor uses canonical upper-triangular boolean adjacency at every period.


Set the project root, import project modules, and configure quiet logging for notebook execution.


In [ ]:
# ================================================================
# CELL 1 · PROJECT SETUP
# ================================================================
import sys, logging, warnings, json
from pathlib import Path

def find_root_with_src(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir():
            return p
    raise RuntimeError("Directory 'src/' not found.")

ROOT = find_root_with_src(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Disable module logging to keep notebook output focused.
logging.disable(logging.CRITICAL)
for name in ("src", "src.model.inference_parameters", "src.preprocessing.core_detection"):
    lg = logging.getLogger(name)
    lg.disabled = True
    lg.propagate = False
    for h in list(lg.handlers):
        lg.removeHandler(h)

warnings.filterwarnings("ignore")

# Project imports
from src.io.load_us_air import load_us_air_snapshots, build_period_maps
from src.analysis import period_summary, time_series
from src.analysis.summary_table import build_small_table

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


Configure the main parameters: data path, period range, core-selection criterion, and Monte Carlo/KS seeds.


In [ ]:
# ================================================================
# CELL 2 · USER CONFIGURATION
# ================================================================
CONFIG = {
    # Data source
    "processed_dir" : ROOT / "data" / "processed" / "us_air",

    # Temporal aggregation
    "time_agg"      : "M",   # 'M' | 'Q' | 'A'
    "chunksize"     : 2_000_000,

    # Period range (1-based)
    "time_start"    : 1,
    "time_end"      : 132,

    # Core-size selection
    "criterion"  : "plsd_diag",   # 'nll' | 'plsd_diag' | 'plsd_maha'
    "complexity_penalty": "none",  # None | 'none' | 'aic' | 'bic'
    "freeze_plsd_variances": True,
    "batch_size" : 12,

    # Parameter estimation
    "tol"        : 1e-8,
    "max_iter"   : 1_000,
    "max_iter_plsd": 200,

    # Monte Carlo and KS
    "R"          : 1000,
    "seed_MC"    : 1234,
    "n_KS"       : 200,
    "seed_KS"    : 5678,
}
print(json.dumps(CONFIG, indent=2, default=str))


Load monthly snapshots, build period metadata maps, define the analysis range, and construct the candidate grid for \(k\).


In [ ]:
# ================================================================
# CELL 3 · DATA LOADING AND PERIOD INDEX
# ================================================================
A_time, period_index = load_us_air_snapshots(
    CONFIG["processed_dir"],
    agg=CONFIG["time_agg"],
    chunksize=CONFIG["chunksize"],
)
period_map, period_ids, period_labels = build_period_maps(period_index)
N, _, n_periods = A_time.shape
print(f"Snapshots ({CONFIG['time_agg']}): {A_time.shape}")

# Candidate core-size values
K_RANGE = list(range(0, N + 1))

# Selected period range (1-based)
time_start = CONFIG["time_start"]
time_end = n_periods if CONFIG["time_end"] is None else CONFIG["time_end"]
if not (1 <= time_start <= time_end <= n_periods):
    raise ValueError("Invalid period range.")

B = CONFIG["batch_size"]
effective_start = max(time_start, B + 1)
sel_periods = range(effective_start, time_end + 1)
print(f"Analyzing periods {effective_start}-{time_end} (count={len(sel_periods)})")


## Core-size selection

For each period, search for the optimal core size \(k\) using NLL or PLSD with optional complexity penalties (AIC/BIC).
The case \(k=0\) corresponds to an Erdos-Renyi baseline with \(p=\sigma(y)\).


## Parameter estimation \(y,\{x_i\}\)

Parameters maximize the selected objective under the model constraints on link counts and core-node degrees.
Optimization uses L-BFGS-B. For PLSD objectives, motif discrepancy penalties are added to the likelihood term.


## Simulations and diagnostics

Use the fitted parameters to simulate synthetic networks with deterministic seeds and compare empirical versus simulated metrics.
Diagnostics include motif statistics, path-based metrics, assortativity, modularity, and KS tests on degree distributions.


## Main checks

Compare empirical and simulated distributions and track consistency over time.


Run the analysis loop over selected periods and store one output row per period.


In [ ]:
# ================================================================
# CELL 4 · MAIN LOOP WITH TQDM
# ================================================================
rows = []

for period_pos in tqdm(sel_periods, desc="Periods", unit="p"):
    row = period_summary.summarize_period(
        A_time,
        period_pos,
        batch_size=CONFIG["batch_size"],
        k_range=K_RANGE,
        criterion=CONFIG["criterion"],
        complexity_penalty=CONFIG["complexity_penalty"],
        freeze_plsd_variances=CONFIG["freeze_plsd_variances"],
        tol=CONFIG["tol"],
        max_iter=CONFIG["max_iter"],
        max_iter_plsd=CONFIG["max_iter_plsd"],
        R=CONFIG["R"],
        seed_MC=CONFIG["seed_MC"],
        n_KS=CONFIG["n_KS"],
        seed_KS=CONFIG["seed_KS"],
        period_map=period_map,
        period_ids=period_ids,
        period_labels=period_labels,
    )
    rows.append(row)

df = pd.DataFrame(rows)

cfg = dict(CONFIG)
cfg["effective_start"] = int(effective_start)
cfg["time_end"] = int(time_end)

import hashlib

config_hash = hashlib.sha1(
    json.dumps(cfg, sort_keys=True, default=str).encode()
).hexdigest()[:10]
run_id = (
    f"agg-{cfg['time_agg']}_t{cfg['time_start']}-{cfg['time_end']}"
    f"_B{cfg['batch_size']}_crit-{cfg['criterion']}"
    f"_pen-{cfg['complexity_penalty']}"
    f"_fr{int(cfg['freeze_plsd_variances'])}_h{config_hash}"
)

run_dir = ROOT / "results" / run_id
tables_dir = run_dir / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

big_path = tables_dir / "big_table.csv"
df.to_csv(big_path, index=False, sep=";", decimal=",")

small_table = build_small_table(df)
small_path = tables_dir / "small_table.csv"
small_table.to_csv(small_path, index=False, sep=";", decimal=",")

config_path = run_dir / "config_resolved.json"
config_path.write_text(json.dumps(cfg, indent=2, default=str))

time_series.plot_series(df, run_id=run_id)
print("Saved:", big_path, small_path, config_path)

df.head()


## Time series and core stability

Generate time series for parameters, metrics, KS p-values, and consecutive core-set Jaccard similarity.
Plots are stored under `figures/`.


Rebuild the plotting DataFrame and export PNG series plots for temporal comparison.


In [ ]:
# ================================================================
# CELL 5 · TIME-SERIES FIGURES
# ================================================================
df_plot = df.reset_index()
time_series.plot_series(df_plot, run_id=run_id)
time_series.plot_degree_distributions(
    A_time,
    df,
    run_id=run_id,
    n_periods=5,
    n_sim=CONFIG["R"],
    seed_MC=CONFIG["seed_MC"],
)
print("Figures saved in", ROOT / "figures" / run_id)


## Result export

Period-level summaries are written to `results/` and figures are written to `figures/`.


## Quick summary

- `CONFIG` controls period range, objective, complexity penalty, optimization options, and simulation seeds.
- `effective_start = max(time_start, batch_size + 1)` defines the first valid period for rolling-window ranking.
- `run_id` includes a short hash of the resolved configuration for reproducible output paths.
